In [22]:
import os
os.environ["POLARS_MAX_THREADS"] = "1"

In [23]:
import polars as pl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

raw_df = pl.scan_csv("../data/Airline_Delay_Cause.csv")
raw_df.head().collect()

year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
i64,i64,str,str,str,str,i64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64
2022,5,"""9E""","""Endeavor Air Inc.""","""ABE""","""Allentown/Bethlehem/Easton, PA…",136,7,5.95,0.0,0.05,0.0,1.0,0,0,255,222,0,4,0,29
2022,5,"""9E""","""Endeavor Air Inc.""","""ABY""","""Albany, GA: Southwest Georgia …",91,16,7.38,0.0,2.54,0.0,6.09,0,0,884,351,0,81,0,452
2022,5,"""9E""","""Endeavor Air Inc.""","""ACK""","""Nantucket, MA: Nantucket Memor…",19,2,0.13,0.0,1.0,0.0,0.88,1,0,138,4,0,106,0,28
2022,5,"""9E""","""Endeavor Air Inc.""","""AEX""","""Alexandria, LA: Alexandria Int…",88,14,7.26,0.76,4.35,0.0,1.64,0,0,947,585,35,125,0,202
2022,5,"""9E""","""Endeavor Air Inc.""","""AGS""","""Augusta, GA: Augusta Regional …",181,19,13.84,0.0,3.07,0.0,2.09,0,0,808,662,0,87,0,59


In [24]:
raw_df.describe()

statistic,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
str,f64,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",318017.0,318017.0,"""318013""","""318013""","""318014""","""318017""",317524.0,317285.0,317525.0,317523.0,317529.0,317529.0,317529.0,317529.0,317527.0,317523.0,317525.0,317529.0,317529.0,317527.0,317529.0
"""null_count""",0.0,0.0,"""4""","""4""","""3""","""0""",493.0,732.0,492.0,494.0,488.0,488.0,488.0,488.0,490.0,494.0,492.0,488.0,488.0,490.0,488.0
"""mean""",2012.450957,6.497844,null,null,null,null,381.76667,72.905076,21.073149,2.616407,24.005228,0.179037,24.975734,7.207257,0.867674,4209.989113,1286.577224,220.567542,1099.516422,7.214845,1596.062993
"""std""",5.678296,3.459423,null,null,null,null,1027.156722,198.936754,47.67158,9.96864,85.113757,0.844834,75.275223,37.216301,3.915772,12519.021012,3515.417309,861.52144,4636.475908,38.854685,4924.950687
"""min""",2003.0,1.0,"""9E""","""ATA Airlines d/b/a ATA""","""ABE""","""Aberdeen, SD: Aberdeen Regiona…",1.0,0.0,0.0,0.0,-0.01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-19.0,0.0,0.0
"""25%""",2007.0,3.0,null,null,null,null,59.0,9.0,3.0,0.0,1.68,0.0,1.64,0.0,0.0,436.0,148.0,0.0,56.0,0.0,79.0
"""50%""",2012.0,6.0,null,null,null,null,120.0,23.0,8.19,0.58,5.49,0.0,5.86,1.0,0.0,1201.0,437.0,25.0,203.0,0.0,351.0
"""75%""",2018.0,10.0,null,null,null,null,273.0,56.0,19.69,2.0,15.35,0.0,17.04,4.0,1.0,3080.0,1100.0,159.0,602.0,0.0,1110.0
"""max""",2022.0,12.0,"""YX""","""Virgin America""","""YUM""","""Yuma, AZ: Yuma MCAS/Yuma Inter…",21977.0,6377.0,1792.07,717.94,4091.27,80.56,1885.47,4951.0,256.0,433687.0,196944.0,57707.0,238440.0,3760.0,148181.0


In [25]:
print(f"Số dòng : {raw_df.collect().height}")

Số dòng : 318017


In [26]:
raw_df.collect_schema().names()

['year',
 'month',
 'carrier',
 'carrier_name',
 'airport',
 'airport_name',
 'arr_flights',
 'arr_del15',
 'carrier_ct',
 'weather_ct',
 'nas_ct',
 'security_ct',
 'late_aircraft_ct',
 'arr_cancelled',
 'arr_diverted',
 'arr_delay',
 'carrier_delay',
 'weather_delay',
 'nas_delay',
 'security_delay',
 'late_aircraft_delay']

# Tìm thời gian

In [27]:
df = raw_df.with_columns(
    pl.date("year", "month", 1).alias("flight_date") 
).drop(["year", "month"])

print(df.select(
    [
        pl.col("flight_date").min().alias("min"),
        pl.col("flight_date").max().alias("max")
    ]
).collect())

shape: (1, 2)
┌────────────┬────────────┐
│ min        ┆ max        │
│ ---        ┆ ---        │
│ date       ┆ date       │
╞════════════╪════════════╡
│ 2003-06-01 ┆ 2022-05-01 │
└────────────┴────────────┘


# Xử lý dữ liệu văn bản

## 1. Carrier

In [28]:
df.select(["carrier", "carrier_name"]).unique().filter(
    (pl.col("carrier").is_null()) | 
    (pl.col("carrier_name").is_null()) 
).collect()

carrier,carrier_name
str,str
null,null


In [29]:
CARRIER_TABLE = df.select(["carrier", "carrier_name"]).unique().drop_nulls()
CARRIER_TABLE.collect()

carrier,carrier_name
str,str
"""DH""","""Atlantic Coast Airlines"""
"""HP""","""America West Airlines Inc."""
"""UA""","""United Air Lines Inc."""
"""MQ""","""Envoy Air"""
"""NK""","""Spirit Air Lines"""
…,…
"""MQ""","""American Eagle Airlines Inc."""
"""G4""","""Allegiant Air"""
"""XE""","""ExpressJet Airlines Inc."""


### Kết luận: Chỉ cần sử dụng bảng CARRIER_TABLE, các giá trị null ở 2 cột carrier và carrier_table bị lược bỏ

In [30]:
df = df.drop_nulls(["carrier", "carrier_name"])

In [31]:
CARRIER_TABLE.collect().write_parquet("../output/carrier.parquet")

## 2. Airport

In [32]:
temp = df.select(["airport", "airport_name"]).unique().filter(
    (pl.col("airport").is_null()) | 
    (pl.col("airport_name").is_null()) 
)

In [33]:
AIRPORT_TABLE = df.select(["airport", "airport_name"]).unique().filter(
    (pl.col("airport").is_not_null()) & 
    (pl.col("airport_name").is_not_null()) 
)

AIRPORT_TABLE.collect()

airport,airport_name
str,str
"""JST""","""Johnstown, PA: John Murtha Joh…"
"""ECP""","""Panama City, FL: Northwest Flo…"
"""PGV""","""Greenville, NC: Pitt Greenvill…"
"""LEX""","""Lexington, KY: Blue Grass"""
"""BIH""","""Bishop, CA: Bishop Airport"""
…,…
"""SGF""","""Springfield, MO: Springfield-B…"
"""ISP""","""Islip, NY: Long Island MacArth…"
"""LBF""","""North Platte, NE: North Platte…"


In [34]:
rescue_check = AIRPORT_TABLE.join(
    temp, 
    on="airport_name", 
    how="semi"
).collect()

print(rescue_check)

shape: (3, 2)
┌─────────┬─────────────────────────────────┐
│ airport ┆ airport_name                    │
│ ---     ┆ ---                             │
│ str     ┆ str                             │
╞═════════╪═════════════════════════════════╡
│ PHX     ┆ Phoenix, AZ: Phoenix Sky Harbo… │
│ PSG     ┆ Petersburg, AK: Petersburg Jam… │
│ PIT     ┆ Pittsburgh, PA: Pittsburgh Int… │
└─────────┴─────────────────────────────────┘


### Kết luận: Chỉ cần fill các giá trị trống bằng bảng AIRPORT_TABLE

In [35]:
df = (
    df.join(
        AIRPORT_TABLE, 
        on="airport_name", 
        how="left", 
        suffix="_fixed" # Tạo cột tạm airport_fixed để đối chiếu
    )
    .with_columns([
        # Nếu cột 'airport' null thì lấy từ 'airport_fixed', không thì giữ nguyên
        pl.col("airport").fill_null(pl.col("airport_fixed"))
    ])
    .drop("airport_fixed") # Xóa cột tạm sau khi đã dùng xong
)

In [36]:
df.head().collect()

carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,flight_date
str,str,str,str,i64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,date
"""9E""","""Endeavor Air Inc.""","""ABE""","""Allentown/Bethlehem/Easton, PA…",136,7,5.95,0.0,0.05,0.0,1.0,0,0,255,222,0,4,0,29,2022-05-01
"""9E""","""Endeavor Air Inc.""","""ABY""","""Albany, GA: Southwest Georgia …",91,16,7.38,0.0,2.54,0.0,6.09,0,0,884,351,0,81,0,452,2022-05-01
"""9E""","""Endeavor Air Inc.""","""ACK""","""Nantucket, MA: Nantucket Memor…",19,2,0.13,0.0,1.0,0.0,0.88,1,0,138,4,0,106,0,28,2022-05-01
"""9E""","""Endeavor Air Inc.""","""AEX""","""Alexandria, LA: Alexandria Int…",88,14,7.26,0.76,4.35,0.0,1.64,0,0,947,585,35,125,0,202,2022-05-01
"""9E""","""Endeavor Air Inc.""","""AGS""","""Augusta, GA: Augusta Regional …",181,19,13.84,0.0,3.07,0.0,2.09,0,0,808,662,0,87,0,59,2022-05-01


In [37]:
AIRPORT_TABLE.collect().write_parquet("../output/airport.parquet")

## 3. Loại bỏ cột "_name" cho dễ đọc

In [38]:
df = df.drop(["carrier_name", "airport_name"])
df.head().collect()

carrier,airport,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,flight_date
str,str,i64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,date
"""9E""","""ABE""",136,7,5.95,0.0,0.05,0.0,1.0,0,0,255,222,0,4,0,29,2022-05-01
"""9E""","""ABY""",91,16,7.38,0.0,2.54,0.0,6.09,0,0,884,351,0,81,0,452,2022-05-01
"""9E""","""ACK""",19,2,0.13,0.0,1.0,0.0,0.88,1,0,138,4,0,106,0,28,2022-05-01
"""9E""","""AEX""",88,14,7.26,0.76,4.35,0.0,1.64,0,0,947,585,35,125,0,202,2022-05-01
"""9E""","""AGS""",181,19,13.84,0.0,3.07,0.0,2.09,0,0,808,662,0,87,0,59,2022-05-01


In [39]:
df.drop_nulls().collect().write_parquet("../output/main_df.parquet")

In [40]:
with open("../output/numeric_columns.txt", "w") as f:
    # Dùng collect_schema().names() để lấy tên cột từ LazyFrame mà không bị warning
    list_col = df.collect_schema().names()
    
    # Lọc các cột cần thiết
    numeric_cols = [col for col in list_col if col not in ["carrier", "airport", "year", "month"]]
    
    f.write(",".join(numeric_cols))

In [41]:
df.select(["carrier", "airport"]).unique().collect().write_parquet("../output/carrier_airport_map.parquet")

# Với dữ liệu số 

1. Nhóm Tổng quan (Chỉ số chính)

- arr_flights: Tổng số chuyến bay dự kiến hạ cánh xuống sân bay đó.
- arr_del15: Tổng số chuyến bay bị trễ (định nghĩa trễ là trên 15 phút).
- arr_cancelled: Số chuyến bay bị hủy.
- arr_diverted: Số chuyến bay phải hạ cánh xuống sân bay khác (chuyển hướng).

2. Nhóm Tần suất trễ (_ct = Count) --> **Có bao nhiêu chuyến bị trễ bởi lý do X?**

- carrier_ct: Trễ do hãng hàng không (bảo trì, phi hành đoàn đến muộn...).
- weather_ct: Trễ do thời tiết xấu.
- nas_ct: Trễ do Hệ thống Hàng không Quốc gia (kiểm soát không lưu, sân bay quá tải...).
- security_ct: Trễ do an ninh (sơ tán nhà ga, kiểm tra an ninh...).
- late_aircraft_ct: Trễ do "hiệu ứng dây chuyền" (máy bay đến muộn từ chuyến trước nên chuyến sau bị chậm).

3. Nhóm Thời gian trễ (_delay = Minutes) --> **Nếu có trễ thì trễ bao lâu?**

- arr_delay: Tổng số phút trễ của tất cả các chuyến bay cộng lại.
- carrier_delay: Số phút trễ do lỗi của hãng.
- weather_delay: Số phút trễ do thời tiết.
- nas_delay: Số phút trễ do hệ thống NAS.
- security_delay: Số phút trễ do an ninh.
- late_aircraft_delay: Số phút trễ tích lũy do máy bay trước đến muộn.